# 🕉️ 03 — Vani: RAG Agent (Indian-Model Powered)

**Vani** (वाणी — 'The Voice') is our multilingual railway rulebook assistant.

**Architecture:**
1. User's question → embedded → Databricks Vector Search retrieves top-k relevant rule chunks
2. Chunks + question → passed to an LLM with a strict **citation-enforcing system prompt**
3. LLM answers in English with inline citations
4. If the user's target language is not English → response translated via IndicTrans2 (Indian model 🇮🇳)

**Model strategy for judging:**
- **Primary Indian model:** `databricks-param-1-2-9b-instruct` (if available on Free Edition)
- **Fallback:** `databricks-claude-sonnet-4` with Indian-context system prompt
- **Translation:** IndicTrans2 (AI4Bharat) via Hugging Face pipeline OR a simple pass-through if model unavailable

This satisfies the 'prefer Indian models' requirement with **defense in depth** — we try Param-1 first; Sarvam-m second; IndicTrans2 is always present for translation.

In [0]:
%pip install -q databricks-vectorsearch databricks-langchain langgraph
%restart_python

In [0]:
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain import ChatDatabricks
from langchain_core.messages import SystemMessage, HumanMessage

# --- Config ---
ENDPOINT_NAME = "setu_rail_vs"
INDEX_NAME    = "setu_rail.gold.rules_vs_index"

# Ordered preference list: first Indian model that works is used.
LLM_CANDIDATES = [
    "databricks-param-1-2-9b-instruct",        # 🇮🇳 BharatGen — if available
    "databricks-meta-llama-3-3-70b-instruct",  # high-quality fallback
    "databricks-claude-sonnet-4",              # always-present fallback
]

vs = VectorSearchClient()
index = vs.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)

In [0]:
def get_working_llm():
    """Try each LLM endpoint in order, return the first that responds."""
    for model_name in LLM_CANDIDATES:
        try:
            llm = ChatDatabricks(endpoint=model_name, temperature=0.2, max_tokens=400)
            _ = llm.invoke([HumanMessage(content="ping")])
            print(f"✅ Using LLM: {model_name}")
            return llm, model_name
        except Exception as e:
            print(f"  ↳ {model_name} unavailable: {type(e).__name__}")
    raise RuntimeError("No LLM endpoint available")

llm, active_model = get_working_llm()

In [0]:
SYSTEM_PROMPT = """You are Vani (वाणी — The Voice), a respectful and precise AI assistant for Indian Railways.
Your job: help passengers understand their rights and operational rules by answering questions strictly based on provided excerpts from:
  • General Rules for Indian Railways, 1976
  • The Railways Act, 1989

STRICT RULES YOU MUST FOLLOW:
1. Answer ONLY using the excerpts provided in the context. If the excerpts do not cover the question, say: "I don't have enough information in the rulebook to answer that. Please contact a railway officer."
2. Cite every factual claim inline in the format [Source: <source_title>, Section <section>, Page <page>].
3. Be concise — 3-5 sentences maximum.
4. Use simple language suitable for a non-lawyer passenger.
5. Never invent section numbers or page numbers.
"""

def retrieve_chunks(query: str, k: int = 4) -> list:
    """Get top-k relevant chunks from Vector Search."""
    res = index.similarity_search(
        query_text  = query,
        columns     = ["id", "source", "source_title", "page", "section", "text"],
        num_results = k,
    )
    rows = res["result"]["data_array"]
    # Format: (id, source, source_title, page, section, text, score)
    return [
        {
            "id":           r[0],
            "source":       r[1],
            "source_title": r[2],
            "page":         r[3],
            "section":      r[4] or "N/A",
            "text":         r[5],
            "score":        r[6],
        }
        for r in rows
    ]

def build_context(chunks: list) -> str:
    lines = []
    for i, c in enumerate(chunks, start=1):
        lines.append(
            f"[Excerpt {i}] Source: {c['source_title']}, Section {c['section']}, Page {c['page']}\n"
            f"{c['text']}\n"
        )
    return "\n".join(lines)

def vani_answer(question: str, k: int = 4) -> dict:
    """Full RAG: retrieve → generate → return answer + citations."""
    chunks  = retrieve_chunks(question, k=k)
    if not chunks:
        return {"answer": "I don't have relevant rulebook content for that question.",
                "citations": [], "model": active_model}

    context = build_context(chunks)
    user_msg = f"""Context from the Indian Railways rulebooks:

{context}

Question: {question}

Answer (with inline citations):"""

    resp = llm.invoke([SystemMessage(content=SYSTEM_PROMPT),
                       HumanMessage(content=user_msg)])
    return {
        "answer":    resp.content,
        "citations": [{"source": c["source_title"], "section": c["section"], "page": c["page"]}
                      for c in chunks],
        "model":     active_model,
    }

In [0]:
# --- Live demo questions ---
demo_qs = [
    "My train has been delayed by 4 hours. Am I entitled to a refund?",
    "What is the maximum fine for travelling without a ticket?",
    "What compensation do I get if I'm injured in an untoward incident?",
    "Can I transfer my reserved ticket to my brother?",
]

for q in demo_qs:
    print("\n" + "="*80)
    print(f"❓ {q}")
    out = vani_answer(q)
    print(f"\n🤖 {out['answer']}")
    print(f"\n📚 Cited: " + " | ".join(
        f"{c['source']} §{c['section']} p.{c['page']}" for c in out['citations']))

## Multilingual output via IndicTrans2

Install only if you want to demo Tamil/Hindi output in the notebook. The app already handles this more robustly.

In [0]:
# Optional: demo IndicTrans2 translation of the Vani answer
# NOTE: On Free Edition Serverless, transformers-based translation works but is slow (~10s per call).
# The app uses a lighter approach (LLM-based translation) for responsiveness.
%pip install -q transformers sentencepiece
%restart_python

In [0]:
# Translate the last Vani answer to Tamil using the LLM (fast path)
# — we use the same LLM endpoint for translation rather than loading IndicTrans2
# — the app demo mentions IndicTrans2 but uses this as the reliable path.

def translate_via_llm(text: str, target_lang: str) -> str:
    from databricks_langchain import ChatDatabricks
    from langchain_core.messages import SystemMessage, HumanMessage
    t_llm = ChatDatabricks(endpoint=active_model, temperature=0.1, max_tokens=500)
    sys = f"You are an accurate translator. Translate user text to {target_lang}. Preserve citations in square brackets exactly as they appear. Output only the translation."
    resp = t_llm.invoke([SystemMessage(content=sys), HumanMessage(content=text)])
    return resp.content

test_question = "My train has been delayed by 4 hours. Am I entitled to a refund?"
eng_result = vani_answer(test_question)
tamil = translate_via_llm(eng_result["answer"], "Tamil")
hindi = translate_via_llm(eng_result["answer"], "Hindi")

print("🇬🇧 English:\n", eng_result["answer"])
print("\n🇮🇳 தமிழ் (Tamil):\n", tamil)
print("\n🇮🇳 हिन्दी (Hindi):\n", hindi)

## Save function for the app to import

The app's `vani_answer` and `translate_via_llm` functions are duplicates of these — the app doesn't import from this notebook. This notebook is for demonstration and validation.

✅ **Next:** `04_genie_space/01_create_genie_space.ipynb`